In [1]:
import pandas as pd
import numpy as np

# load dataset
df = pd.read_csv("weatherAUS.csv")

# clean column names
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
df.head()


Shape: (145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [2]:
# check duplicates
print("Full row duplicates:", df.duplicated().sum())

# missing values (top)
missing = df.isna().sum().sort_values(ascending=False)
print("Top missing columns:")
print(missing.head(10))


Full row duplicates: 0
Top missing columns:
Sunshine         69835
Evaporation      62790
Cloud3pm         59358
Cloud9am         55888
Pressure9am      15065
Pressure3pm      15028
WindDir9am       10566
WindGustDir      10326
WindGustSpeed    10263
Humidity3pm       4507
dtype: int64


In [3]:
# convert to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# create simple date features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek


In [4]:
# remove rows with missing target
df = df.dropna(subset=["RainTomorrow"]).reset_index(drop=True)

# encode target: Yes=1, No=0
df["RainTomorrow"] = df["RainTomorrow"].map({"Yes": 1, "No": 0})

print("Target distribution:")
print(df["RainTomorrow"].value_counts())


Target distribution:
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64


In [5]:
# RainToday has missing values, fill first then encode
df["RainToday"] = df["RainToday"].fillna("No")
df["RainToday"] = df["RainToday"].map({"Yes": 1, "No": 0})

# separate columns
target_col = "RainTomorrow"

# columns to drop (not needed after feature extraction)
drop_cols = ["Date"]

# categorical features
cat_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm"]

# numeric features
num_cols = [c for c in df.columns if c not in cat_cols + drop_cols + [target_col]]

print("Numeric cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))


Numeric cols: 20
Categorical cols: 4


In [6]:
# replace inf values if any
df = df.replace([np.inf, -np.inf], np.nan)

# fill numeric missing with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# fill categorical missing with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# final missing check
print("Missing values after preprocessing:", df.isna().sum().sum())



Missing values after preprocessing: 0
